# Fine-tuning del rilevatore di racchetta

Parte dal modello attuale (`tennis_yolo11.pt`) e lo riaddestra sui nostri frame annotati.

**Prima di eseguire** serve su Drive, in **Il mio Drive/rf_coach_vision/dataset/**:
`images/` (i frame) e `labels/` (le etichette YOLO **corrette a mano**).

Ordine del lavoro: `estrai_frame.py` sul PC → `preannota.py` (qui su Colab, cella 5) →
correzione su Roboflow o CVAT → rimetti `images/` e `labels/` su Drive → addestramento.

Runtime → Cambia tipo di runtime → **GPU T4**.

In [ ]:
!nvidia-smi -L

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## Codice da GitHub e dati da Drive

In [ ]:
import os, shutil, glob, subprocess, random

REPO = "anrundo-2312/rf_coach_vision"
DRIVE_DIR = "/content/drive/MyDrive/rf_coach_vision"
WORK_DIR = "/content/rf_coach_vision"

token = ""
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN") or ""
except Exception:
    pass

url = f"https://{token}@github.com/{REPO}.git" if token else f"https://github.com/{REPO}.git"
if os.path.exists(WORK_DIR):
    shutil.rmtree(WORK_DIR)
subprocess.run(["git", "clone", "--depth", "1", url, WORK_DIR], check=True)

for sub, dst in [("models", ""), ("dataset/images", "finetuning/dataset/images"),
                 ("dataset/labels", "finetuning/dataset/labels")]:
    src = os.path.join(DRIVE_DIR, sub)
    dst_dir = os.path.join(WORK_DIR, dst) if dst else WORK_DIR
    os.makedirs(dst_dir, exist_ok=True)
    if os.path.isdir(src):
        for f in glob.glob(os.path.join(src, "*")):
            if os.path.isfile(f):
                shutil.copy(f, dst_dir)

img = glob.glob(os.path.join(WORK_DIR, "finetuning/dataset/images/*.jpg"))
lbl = glob.glob(os.path.join(WORK_DIR, "finetuning/dataset/labels/*.txt"))
print(f"immagini: {len(img)}   etichette: {len(lbl)}")

In [ ]:
!pip install -q ultralytics

## (Solo la prima volta) Pre-annotazione

Da eseguire quando su Drive ci sono le immagini ma non ancora le etichette.
Produce riquadri di partenza da correggere a mano. Poi scarica la cartella
`finetuning/dataset/labels` (pannello file a sinistra) e caricala su Roboflow o CVAT
insieme alle immagini.

Se le etichette corrette ci sono già, **salta questa cella**.

In [ ]:
%cd {WORK_DIR}
!python finetuning/preannota.py --conf 0.15 --anteprime

## Divisione train/validazione

La divisione è **per video**, non casuale: frame dello stesso scambio si somigliano
troppo, e se finissero metà in addestramento e metà in validazione il punteggio
sarebbe gonfiato. Teniamo da parte un video intero.

In [ ]:
VIDEO_VALIDAZIONE = "nicola_matarese_trim"   # il video tenuto fuori dall'addestramento

base = os.path.join(WORK_DIR, "finetuning/dataset")
for split in ("train", "val"):
    for kind in ("images", "labels"):
        os.makedirs(os.path.join(base, split, kind), exist_ok=True)

n = {"train": 0, "val": 0}
for img_path in sorted(glob.glob(os.path.join(base, "images", "*.jpg"))):
    nome = os.path.splitext(os.path.basename(img_path))[0]
    lbl_path = os.path.join(base, "labels", nome + ".txt")
    if not os.path.exists(lbl_path):
        continue
    split = "val" if nome.startswith(VIDEO_VALIDAZIONE) else "train"
    shutil.copy(img_path, os.path.join(base, split, "images"))
    shutil.copy(lbl_path, os.path.join(base, split, "labels"))
    n[split] += 1

yaml_path = os.path.join(base, "dataset.yaml")
open(yaml_path, "w").write(f"path: {base}\ntrain: train/images\nval: val/images\nnames:\n  0: racket\n")
print(n, "->", yaml_path)
assert n["train"] and n["val"], "Servono immagini in entrambe le parti: controlla VIDEO_VALIDAZIONE"

## Addestramento

`imgsz=960` come nell'analisi. Poche epoche bastano: partiamo da un modello già addestrato,
e con poche centinaia di immagini si rischia presto di imparare a memoria.
`patience` ferma tutto se per 15 epoche non migliora.

In [ ]:
EPOCHE = 60

from ultralytics import YOLO
modello = YOLO(os.path.join(WORK_DIR, "tennis_yolo11.pt"))
risultati = modello.train(data=yaml_path, epochs=EPOCHE, imgsz=960, batch=8,
                          patience=15, project=os.path.join(WORK_DIR, "runs"), name="racchetta")

## Confronto: modello vecchio contro modello nuovo, sullo stesso video di validazione

In [ ]:
nuovo = os.path.join(WORK_DIR, "runs/racchetta/weights/best.pt")

for nome, pesi in [("vecchio", os.path.join(WORK_DIR, "tennis_yolo11.pt")), ("nuovo", nuovo)]:
    m = YOLO(pesi)
    r = m.val(data=yaml_path, imgsz=960, classes=[0], verbose=False)
    print(f"{nome:8s}  mAP50={r.box.map50:.3f}  mAP50-95={r.box.map:.3f}  "
          f"precisione={r.box.mp:.3f}  richiamo={r.box.mr:.3f}")

mAP50 e mAP50-95 misurano quanto i riquadri previsti coprono quelli veri.
**Richiamo** = quante racchette vere trova (il nostro problema principale: oggi ne perde due su tre).
**Precisione** = quante delle racchette trovate sono vere (ombre e racchette altrui).

## Salva il modello nuovo su Drive

In [ ]:
dst = os.path.join(DRIVE_DIR, "models")
os.makedirs(dst, exist_ok=True)
shutil.copy(nuovo, os.path.join(dst, "tennis_yolo11_finetuned.pt"))
print("salvato in", os.path.join(dst, "tennis_yolo11_finetuned.pt"))
print("Per usarlo nell'analisi: mettilo in rfCoach_vision e cambia il nome del file in analyze.py")